In [1]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [2]:
df['label'] = df['sentiment'].map({
    'negative': 0,
    'positive': 1
})

In [3]:
df[['review', 'label']].head()

,review,label
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [4]:
from sklearn.model_selection import train_test_split

# Reduce dataset
df = df.sample(5000, random_state=42)

# First split (train + temp)
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df['review'],
    df['label'],
    test_size=0.2,
    random_state=42
)

# Second split (validation + test)
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.5,
    random_state=42
)

In [5]:
from transformers import RobertaTokenizer

tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [6]:
train_encodings = tokenizer(
    list(train_texts),
    truncation=True,
    padding=True,
    max_length=48
)

test_encodings = tokenizer(
    list(test_texts),
    truncation=True,
    padding=True,
    max_length=48
)

In [7]:
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [8]:
train_dataset = IMDbDataset(train_encodings, train_labels)
test_dataset = IMDbDataset(test_encodings, test_labels)

In [9]:
import torch.nn as nn
from transformers import RobertaModel

class Model(nn.Module):
    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained('roberta-base')

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            batch_first=True,
            bidirectional=True
        )

        self.projection = nn.Linear(512, 768)

        self.dropout = nn.Dropout(0.3)

        self.fc = nn.Linear(768, 2)

    def forward(self, input_ids, attention_mask):

        roberta_out = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = roberta_out.last_hidden_state

        lstm_out, _ = self.lstm(sequence_output)

        # Mean pooling
        lstm_feat = torch.mean(lstm_out, dim=1)
        roberta_feat = torch.mean(sequence_output, dim=1)

        # Residual connection
        combined = roberta_feat + self.projection(lstm_feat)

        out = self.dropout(combined)

        return self.fc(out)

In [10]:
from torch.utils.data import DataLoader
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Model().to(device)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=2e-5)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
for epoch in range(3):

    model.train()

    for batch in train_loader:

        optimizer.zero_grad()

        outputs = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        )

        loss = criterion(outputs, batch['labels'].to(device))

        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1} done")

Epoch 1 done
Epoch 2 done
Epoch 3 done


In [12]:
from sklearn.metrics import classification_report

model.eval()

predictions = []
true_labels = []

with torch.no_grad():

    for batch in DataLoader(test_dataset, batch_size=16):

        outputs = model(
            batch['input_ids'].to(device),
            batch['attention_mask'].to(device)
        )

        preds = torch.argmax(outputs, dim=1)

        predictions.extend(preds.cpu().tolist())
        true_labels.extend(batch['labels'].tolist())

print(classification_report(true_labels, predictions))

              precision    recall  f1-score   support

           0       0.74      0.90      0.81       252
           1       0.87      0.68      0.76       248

    accuracy                           0.79       500
   macro avg       0.80      0.79      0.79       500
weighted avg       0.80      0.79      0.79       500

